## Cell 1 — Load Weighted Encodings

Loads all `real_*.txt` files, builds:

- `xb` → full matrix `(N, D)`
- `poly_id_to_idx`
- `idx_to_poly_id`

Also prints load time, shape, memory.

In [ ]:
import os, glob, re, time, numpy as np
from tqdm.auto import tqdm

ENC_DIR = "/raid/ruban/encodings/pk-real10k0.002" 
GT_DIR = "/raid/ruban/groundtruth/pk-query-10k"

print("Loading weighted encodings...")
t0 = time.time()

files = sorted(
    glob.glob(os.path.join(ENC_DIR, "real_*.txt")),
    key=lambda x: int(re.search(r"real_(\d+)", x).group(1))
)

xb_list = []
poly_id_to_idx = {}
idx_to_poly_id = {}

idx = 0

for fp in tqdm(files, desc="Files"):
    start_id = int(re.search(r"real_(\d+)", fp).group(1))

    with open(fp, "r") as f:
        for row_offset, line in enumerate(f):
            poly_id = start_id + row_offset
            vals = np.fromstring(line.strip(), sep=" ", dtype=np.float32)

            xb_list.append(vals)
            poly_id_to_idx[poly_id] = idx
            idx_to_poly_id[idx] = poly_id
            idx += 1

xb = np.vstack(xb_list).astype(np.float32)

dt = time.time() - t0
mem_gb = xb.nbytes / (1024**3)

print(f"Loaded polygons : {len(xb)}")
print(f"Matrix shape    : {xb.shape}")
print(f"Memory (GB)     : {mem_gb:.3f}")
print(f"Load time (s)   : {dt:.2f}")


def make_topk_matrix(x, K):
    out = np.zeros_like(x, dtype=np.float32)

    for i in tqdm(range(len(x)), desc=f"Building Top-{K}"):
        row = x[i]
        nz_idx = np.flatnonzero(row > 0)

        if len(nz_idx) <= K:
            out[i, nz_idx] = row[nz_idx]
            continue

        vals = row[nz_idx]
        top_local = np.argpartition(vals, -K)[-K:]
        keep_idx = nz_idx[top_local]

        out[i, keep_idx] = row[keep_idx]

    return out

t0 = time.time()

xb_top2048 = make_topk_matrix(xb, 2048)

xb = xb_top2048
print("Using Top2048 matrix:", xb.shape)

Loading weighted encodings...


Files:   0%|          | 0/80 [00:00<?, ?it/s]

Loaded polygons : 10000
Matrix shape    : (10000, 18499)
Memory (GB)     : 0.689
Load time (s)   : 50.40
Using Top2048 matrix: (10000, 18499)


## Cell 2 — Load Ground Truth + Split DB / Queries

Assumes last 20% IDs are query side.

Builds:

- `gt`
- `GT_ID_THRESHOLD`
- `db_indices`
- `q_indices`
- `xdb`
- `xq`

In [18]:
import glob, os, time
from tqdm.auto import tqdm

print("Loading ground truth...")
t0 = time.time()

gt = {}

gt_files = sorted(glob.glob(os.path.join(GT_DIR, "similarityMap_*")))

for fp in tqdm(gt_files, desc="GT Files"):
    with open(fp, "r") as f:
        for line in f:
            ids = [int(x.strip()) for x in line.strip().split(",") if x.strip()]
            if ids:
                gt[ids[0]] = set(ids[1:])

print(f"Loaded GT queries : {len(gt)}")

# ---- infer threshold from dataset size (80/20 split) ----
all_poly_ids = sorted(poly_id_to_idx.keys())
GT_ID_THRESHOLD = all_poly_ids[int(len(all_poly_ids) * 0.80)]

print(f"GT_ID_THRESHOLD   : {GT_ID_THRESHOLD}")

# ---- split ----
db_indices = []
q_indices  = []

for i in tqdm(range(len(xb)), desc="Splitting"):
    poly_id = idx_to_poly_id[i]

    if poly_id < GT_ID_THRESHOLD:
        db_indices.append(i)
    elif poly_id in gt:
        q_indices.append(i)

xdb = xb[db_indices]
xq  = xb[q_indices]

print(f"DB size    : {len(xdb)}")
print(f"Query size : {len(xq)}")
print(f"xdb shape  : {xdb.shape}")
print(f"xq shape   : {xq.shape}")
print(f"Done in {time.time()-t0:.2f}s")

Loading ground truth...


GT Files:   0%|          | 0/60 [00:00<?, ?it/s]

Loaded GT queries : 2000
GT_ID_THRESHOLD   : 8000


Splitting:   0%|          | 0/10000 [00:00<?, ?it/s]

DB size    : 8000
Query size : 2000
xdb shape  : (8000, 18499)
xq shape   : (2000, 18499)
Done in 0.26s


## Cell 3 — Environment Check + Normalize Inputs

Checks GPU availability and L2-normalizes vectors.

Why:
For first experiment, we use your weighted vectors as input features and learn dense embeddings from them.
Normalization usually helps training stability.

In [19]:
import torch, time
from sklearn.preprocessing import normalize

print("Torch version :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    device = "cuda"
else:
    device = "cpu"

t0 = time.time()

xdb_norm = normalize(xdb, norm="l2", axis=1).astype(np.float32)
xq_norm  = normalize(xq,  norm="l2", axis=1).astype(np.float32)

print("xdb_norm shape:", xdb_norm.shape)
print("xq_norm shape :", xq_norm.shape)
print(f"Normalization done in {time.time()-t0:.2f}s")
print("Device:", device)

Torch version : 2.5.1+cu121
CUDA available: True
GPU : NVIDIA A100-SXM4-80GB
xdb_norm shape: (8000, 18499)
xq_norm shape : (2000, 18499)
Normalization done in 0.33s
Device: cuda


## Cell 4 — Build Training Pairs from Ground Truth

Creates positive training pairs:

(anchor_db_idx, positive_db_idx)

We only use GT neighbors that exist inside DB (< threshold).

This gives supervised signal immediately.

In [6]:
import time
from tqdm.auto import tqdm

print("Building supervised query->DB pairs...")
t0 = time.time()

# fast maps
global_to_db_local = {g:i for i, g in enumerate(db_indices)}
global_to_q_local  = {g:i for i, g in enumerate(q_indices)}

train_pairs = []

for q_pid, nbrs in tqdm(gt.items(), desc="GT Queries"):
    
    if q_pid not in poly_id_to_idx:
        continue
        
    q_global = poly_id_to_idx[q_pid]
    
    if q_global not in global_to_q_local:
        continue
        
    q_local = global_to_q_local[q_global]

    for nb_pid in nbrs:
        if nb_pid in poly_id_to_idx:
            nb_global = poly_id_to_idx[nb_pid]

            if nb_global in global_to_db_local:
                db_local = global_to_db_local[nb_global]
                train_pairs.append((q_local, db_local))
                break

print(f"Training pairs : {len(train_pairs)}")
print(f"Coverage       : {len(train_pairs)/len(gt):.3f}")
print(f"Done in {time.time()-t0:.2f}s")

Building supervised query->DB pairs...


GT Queries:   0%|          | 0/2000 [00:00<?, ?it/s]

Training pairs : 1818
Coverage       : 0.909
Done in 0.01s


## Cell 5 — Train Dense Embedding Model (Fast MLP)

Learns:

18499 dims → 512 → 128 embedding

Uses contrastive InfoNCE-style loss:
query embedding should match its GT positive DB embedding.

This is Experiment 1 Challenger.

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import time, numpy as np

EMB_DIM = 128
BATCH   = 256
EPOCHS  = 10
LR      = 1e-3
TEMP    = 0.07

# ---------- dataset ----------
class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        q_i, d_i = self.pairs[idx]
        return xq_norm[q_i], xdb_norm[d_i]

loader = DataLoader(
    PairDataset(train_pairs),
    batch_size=BATCH,
    shuffle=True,
    drop_last=True
)

# ---------- model ----------
class EmbedNet(nn.Module):
    def __init__(self, in_dim, emb_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Linear(512, emb_dim)
        )

    def forward(self, x):
        z = self.net(x)
        return F.normalize(z, dim=1)

model = EmbedNet(xdb_norm.shape[1], EMB_DIM).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)

# ---------- train ----------
print("Training embedding model...")
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    losses = []

    bar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for q_batch, d_batch in bar:
        q_batch = q_batch.to(device)
        d_batch = d_batch.to(device)

        zq = model(q_batch)
        zd = model(d_batch)

        logits = (zq @ zd.T) / TEMP
        labels = torch.arange(len(zq), device=device)

        loss = F.cross_entropy(logits, labels)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
        bar.set_postfix(loss=np.mean(losses))

print(f"Training done in {time.time()-t0:.1f}s")

Training embedding model...


Epoch 1/10:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 2/10:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 3/10:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 4/10:   0%|          | 0/7 [00:00<?, ?it/s]

...

Epoch 7/10:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 8/10:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 9/10:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 10/10:   0%|          | 0/7 [00:00<?, ?it/s]

Training done in 1.9s


## Cell 6 — Encode All Vectors with Learned Model

Creates:

- `db_emb`
- `q_emb`

These are the dense vectors for HNSW search.

In [8]:
import numpy as np
import torch
from tqdm.auto import tqdm
import time

def encode_batches(arr, model, batch_size=512):
    model.eval()
    outs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(arr), batch_size), desc="Encoding"):
            batch = torch.tensor(arr[i:i+batch_size], dtype=torch.float32, device=device)
            z = model(batch).cpu().numpy()
            outs.append(z)

    return np.vstack(outs).astype(np.float32)

print("Encoding DB + Queries...")
t0 = time.time()

db_emb = encode_batches(xdb_norm, model)
q_emb  = encode_batches(xq_norm,  model)

print("db_emb shape:", db_emb.shape)
print("q_emb shape :", q_emb.shape)
print(f"Done in {time.time()-t0:.2f}s")

Encoding DB + Queries...


Encoding:   0%|          | 0/16 [00:00<?, ?it/s]

Encoding:   0%|          | 0/4 [00:00<?, ?it/s]

db_emb shape: (8000, 128)
q_emb shape : (2000, 128)
Done in 0.10s


## Cell 7 — Build Global HNSW (hnswlib) + Recall Test

Uses cosine similarity on learned embeddings.

This is cleaner than FAISS for now.

In [11]:
import hnswlib, time, numpy as np
from tqdm.auto import tqdm

TOPK = 50
DIM  = db_emb.shape[1]

print("Building HNSW index...")
t0 = time.time()

index = hnswlib.Index(space='cosine', dim=DIM)

index.init_index(
    max_elements=len(db_emb),
    ef_construction=200,
    M=32
)

index.add_items(db_emb, np.arange(len(db_emb)))
index.set_ef(128)

build_time = time.time() - t0
print(f"Build time: {build_time:.2f}s")

# ---- search ----
print("Running ANN search...")
t1 = time.time()

labels, distances = index.knn_query(q_emb, k=TOPK)

query_time = time.time() - t1
qps = len(q_emb) / query_time

print(f"QPS: {qps:.2f}")

# ---- recall ----
hits = 0
total = 0

for qi in tqdm(range(len(q_indices)), desc="Evaluating"):
    q_global = q_indices[qi]
    q_pid = idx_to_poly_id[q_global]

    if q_pid not in gt:
        continue

    gt_set = gt[q_pid]

    pred_ids = [
        idx_to_poly_id[db_indices[j]]
        for j in labels[qi]
    ]

    hits += len(set(pred_ids) & gt_set)
    total += min(TOPK, len(gt_set))

recall = hits / total if total else 0.0

print(f"\n=== GEO2VEC STYLE PILOT RESULT ===")
print(f"Build Time      : {build_time:.2f}s")
print(f"QPS             : {qps:.2f}")
print(f"Raw Recall@50   : {recall:.4f}")

Building HNSW index...
Build time: 0.39s
Running ANN search...
QPS: 96280.23


Evaluating:   0%|          | 0/2000 [00:00<?, ?it/s]


=== GEO2VEC STYLE PILOT RESULT ===
Build Time      : 0.39s
QPS             : 96280.23
Raw Recall@50   : 0.0188


## Cell 8 — PCA Compression + Dense Retrieval Benchmark

This tests whether your weighted vectors can be compressed while preserving retrieval.

We will:
- reduce `xdb_norm` and `xq_norm` to 256D with PCA
- L2 normalize the compressed vectors
- build one global HNSW
- measure raw Recall@50 and QPS

If this also fails badly, dense compression is the issue, not just the tiny MLP.

In [12]:
import time, numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
import hnswlib

PCA_DIM = 256
TOPK = 50

print(f"Running PCA -> {PCA_DIM} dims ...")
t0 = time.time()

pca = PCA(n_components=PCA_DIM, random_state=42)
xdb_pca = pca.fit_transform(xdb_norm).astype(np.float32)
xq_pca  = pca.transform(xq_norm).astype(np.float32)

xdb_pca = normalize(xdb_pca, norm="l2", axis=1).astype(np.float32)
xq_pca  = normalize(xq_pca,  norm="l2", axis=1).astype(np.float32)

print(f"PCA done in {time.time()-t0:.2f}s")
print("xdb_pca shape:", xdb_pca.shape)
print("xq_pca shape :", xq_pca.shape)

print("\nBuilding PCA HNSW...")
t1 = time.time()

pca_index = hnswlib.Index(space='cosine', dim=PCA_DIM)
pca_index.init_index(max_elements=len(xdb_pca), ef_construction=200, M=32)
pca_index.add_items(xdb_pca, np.arange(len(xdb_pca)))
pca_index.set_ef(128)

build_time = time.time() - t1
print(f"Build time: {build_time:.2f}s")

print("\nRunning PCA ANN search...")
t2 = time.time()
labels_pca, distances_pca = pca_index.knn_query(xq_pca, k=TOPK)
query_time = time.time() - t2
qps = len(xq_pca) / query_time

print(f"QPS: {qps:.2f}")

hits = 0
total = 0

for qi in tqdm(range(len(q_indices)), desc="Evaluating PCA"):
    q_global = q_indices[qi]
    q_pid = idx_to_poly_id[q_global]

    if q_pid not in gt:
        continue

    gt_set = gt[q_pid]
    pred_ids = [idx_to_poly_id[db_indices[j]] for j in labels_pca[qi]]

    hits += len(set(pred_ids) & gt_set)
    total += min(TOPK, len(gt_set))

pca_recall = hits / total if total else 0.0

print(f"\n=== PCA PILOT RESULT ({PCA_DIM}D) ===")
print(f"Build Time    : {build_time:.2f}s")
print(f"QPS           : {qps:.2f}")
print(f"Raw Recall@50 : {pca_recall:.4f}")

Running PCA -> 256 dims ...
PCA done in 3.51s
xdb_pca shape: (8000, 256)
xq_pca shape : (2000, 256)

Building PCA HNSW...
Build time: 0.48s

Running PCA ANN search...
QPS: 104579.16


Evaluating PCA:   0%|          | 0/2000 [00:00<?, ?it/s]


=== PCA PILOT RESULT (256D) ===
Build Time    : 0.48s
QPS           : 104579.16
Raw Recall@50 : 0.0265


## Cell 9 — Analyze Weight Sparsity and Tiny-Value Mass

Before pruning, measure:

- average nonzeros per vector
- percentile nonzeros
- how much total weight lives in tiny values

This tells us whether threshold pruning has a real chance.

In [13]:
import numpy as np
from tqdm.auto import tqdm

THRESHOLDS = [0.0, 1e-6, 1e-5, 1e-4, 1e-3]

print("Analyzing sparsity / tiny-weight mass...")

nnz_stats = []
mass_stats = {t: [] for t in THRESHOLDS}

for i in tqdm(range(len(xb)), desc="Scanning xb"):
    row = xb[i]
    abs_row = np.abs(row)

    nnz = np.count_nonzero(row > 0)
    nnz_stats.append(nnz)

    total_mass = row.sum()
    if total_mass <= 0:
        for t in THRESHOLDS:
            mass_stats[t].append(0.0)
        continue

    for t in THRESHOLDS:
        kept_mass = row[row > t].sum()
        mass_stats[t].append(float(kept_mass / total_mass))

nnz_stats = np.array(nnz_stats)

print("\n=== NNZ Stats ===")
print(f"mean nnz   : {nnz_stats.mean():.2f}")
print(f"median nnz : {np.median(nnz_stats):.2f}")
print(f"p90 nnz    : {np.percentile(nnz_stats, 90):.2f}")
print(f"p95 nnz    : {np.percentile(nnz_stats, 95):.2f}")
print(f"max nnz    : {nnz_stats.max()}")

print("\n=== Mass Kept After Thresholding ===")
for t in THRESHOLDS:
    vals = np.array(mass_stats[t])
    print(f"thr={t:<7g} mean={vals.mean():.6f} median={np.median(vals):.6f} p10={np.percentile(vals,10):.6f}")

Analyzing sparsity / tiny-weight mass...


Scanning xb:   0%|          | 0/10000 [00:00<?, ?it/s]


=== NNZ Stats ===
mean nnz   : 4773.39
median nnz : 3838.50
p90 nnz    : 10388.50
p95 nnz    : 12359.70
max nnz    : 18484

=== Mass Kept After Thresholding ===
thr=0       mean=1.000000 median=1.000000 p10=1.000000
thr=1e-06   mean=0.003364 median=0.000000 p10=0.000000
thr=1e-05   mean=0.000774 median=0.000000 p10=0.000000
thr=0.0001  mean=0.000323 median=0.000000 p10=0.000000
thr=0.001   mean=0.000077 median=0.000000 p10=0.000000


## Cell 10 — Build Top-K Per Row Compressed Matrices

Keeps only strongest weights in each row.

We first measure retained mass for:

- K = 256
- K = 512
- K = 1024
- K = 2048

If mass stays high, retrieval may survive.

In [14]:
import numpy as np
from tqdm.auto import tqdm

TOPK_LIST = [256, 512, 1024, 2048]

print("Evaluating Top-K row retention...")

results = {}

for K in TOPK_LIST:
    kept_mass = []
    kept_nnz  = []

    for i in tqdm(range(len(xb)), desc=f"K={K}"):
        row = xb[i]
        total = row.sum()

        nz_idx = np.flatnonzero(row > 0)

        if len(nz_idx) <= K:
            kept_mass.append(1.0 if total > 0 else 0.0)
            kept_nnz.append(len(nz_idx))
            continue

        vals = row[nz_idx]

        top_local = np.argpartition(vals, -K)[-K:]
        top_vals = vals[top_local]

        ratio = top_vals.sum() / total if total > 0 else 0.0

        kept_mass.append(float(ratio))
        kept_nnz.append(K)

    results[K] = (
        np.mean(kept_mass),
        np.median(kept_mass),
        np.percentile(kept_mass, 10),
        np.mean(kept_nnz)
    )

print("\n=== Top-K Retained Mass ===")
for K in TOPK_LIST:
    a,b,c,d = results[K]
    print(f"K={K:<4} mean={a:.4f} median={b:.4f} p10={c:.4f} avg_nnz={d:.1f}")

Evaluating Top-K row retention...


K=256:   0%|          | 0/10000 [00:00<?, ?it/s]

K=512:   0%|          | 0/10000 [00:00<?, ?it/s]

K=1024:   0%|          | 0/10000 [00:00<?, ?it/s]

K=2048:   0%|          | 0/10000 [00:00<?, ?it/s]


=== Top-K Retained Mass ===
K=256  mean=0.2997 median=0.2178 p10=0.1283 avg_nnz=252.4
K=512  mean=0.4443 median=0.3497 p10=0.2224 avg_nnz=494.6
K=1024 mean=0.6201 median=0.5378 p10=0.3716 avg_nnz=947.3
K=2048 mean=0.7985 median=0.7737 p10=0.5824 avg_nnz=1731.9


## Cell 11 — Build Top-K Compressed Matrices

Creates:

- `xb_top1024`
- `xb_top2048`

Same shape as xb, but only strongest K weights retained per row.
Ready for your existing weighted pipeline.

## Cell 12 — Quick Weighted Jaccard Fidelity Test

Before full pipeline rerun, test whether Top-K preserves true similarity.

Compares original weighted Jaccard vs Top-K weighted Jaccard on random query/db pairs.

If correlation is strong, rerun pipeline is worth it.

Sampling pairs:   0%|          | 0/2000 [00:00<?, ?it/s]

=== Fidelity Test ===
Top1024 corr : 0.8137 | MAE: 0.1028
Top2048 corr : 0.9009 | MAE: 0.0722
